In [18]:
import os
import json
from typing import Dict, Any, List, TypedDict
from langchain_openai import ChatOpenAI
from langchain_classic.schema import Document
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
from IPython.display import Image, display
from dotenv import load_dotenv


In [4]:
# ✅ Get the OpenAI API key from .env
openai_api_key = os.getenv("OPENAI_API_KEY")

In [6]:
# Define the state structure
class ResearchState(TypedDict):
    question: str
    sub_questions: List[str]
    search_results: Dict[str, List[Document]]
    analysis: Dict[str, str]
    final_report: str
    current_step: str



In [8]:
# Initialize models and tools
llm = ChatOpenAI(model="gpt-4", temperature=0.7)
search_tool = DuckDuckGoSearchRun()

In [10]:
def question_analyzer_node(state: ResearchState) -> ResearchState:
    """Analyze the research question and break it down into sub-questions"""
    print("🔍 Analyzing research question...")
    
    prompt = f"""
    Analyze the following research question and break it down into 3-5 key sub-questions 
    that would help in conducting comprehensive research.
    
    Research Question: {state['question']}
    
    Return ONLY a JSON array of sub-questions without any additional text.
    
    Example: ["sub-question 1", "sub-question 2", "sub-question 3"]
    """
    
    response = llm.invoke(prompt)
    
    try:
        # Extract JSON from response
        sub_questions = json.loads(response.content)
    except:
        # Fallback if JSON parsing fails
        sub_questions = [
            f"What is {state['question']}?",
            f"Why is {state['question']} important?",
            f"How does {state['question']} work?",
            f"What are the applications of {state['question']}?"
        ]
    
    return {
        **state,
        "sub_questions": sub_questions,
        "current_step": "question_analyzed"
    }


In [11]:
def research_node(state: ResearchState) -> ResearchState:
    """Conduct research for each sub-question"""
    print("📚 Conducting research...")
    
    search_results = {}
    
    for i, sub_question in enumerate(state['sub_questions']):
        print(f"  Researching: {sub_question}")
        
        try:
            # Search for information
            search_result = search_tool.run(sub_question)
            
            # Create document objects for consistency
            search_results[sub_question] = [
                Document(
                    page_content=search_result,
                    metadata={"source": "web_search", "sub_question": sub_question}
                )
            ]
        except Exception as e:
            print(f"  Search failed for '{sub_question}': {e}")
            search_results[sub_question] = [
                Document(
                    page_content="No information found.",
                    metadata={"source": "error", "sub_question": sub_question}
                )
            ]
    
    return {
        **state,
        "search_results": search_results,
        "current_step": "research_completed"
    }

In [12]:
def analysis_node(state: ResearchState) -> ResearchState:
    """Analyze the research results and extract key insights"""
    print("📊 Analyzing research findings...")
    
    analysis = {}
    
    for sub_question, documents in state['search_results'].items():
        if documents and documents[0].page_content != "No information found.":
            prompt = f"""
            Based on the following research information, provide a comprehensive analysis 
            for the sub-question: "{sub_question}"
            
            Research Information:
            {documents[0].page_content[:3000]}  # Limit content length
            
            Provide a well-structured analysis that includes:
            1. Key findings
            2. Important facts
            3. Relevant context
            4. Potential implications
            
            Keep your response concise but informative.
            """
            
            response = llm.invoke(prompt)
            analysis[sub_question] = response.content
        else:
            analysis[sub_question] = "No relevant information found for this sub-question."
    
    return {
        **state,
        "analysis": analysis,
        "current_step": "analysis_completed"
    }

In [13]:
def report_generation_node(state: ResearchState) -> ResearchState:
    """Generate a comprehensive research report"""
    print("📝 Generating final report...")
    
    # Prepare analysis content for report generation
    analysis_content = "\n\n".join([
        f"## {sub_question}\n{analysis}" 
        for sub_question, analysis in state['analysis'].items()
    ])
    
    prompt = f"""
    Based on the comprehensive research and analysis, create a well-structured research report 
    on the topic: "{state['question']}"
    
    Research Analysis:
    {analysis_content}
    
    Structure your report as follows:
    
    # Research Report: {state['question']}
    
    ## Executive Summary
    Provide a brief overview of key findings.
    
    ## Detailed Analysis
    Present the comprehensive analysis in a structured manner.
    
    ## Key Insights
    Highlight the most important discoveries.
    
    ## Conclusion
    Summarize the findings and suggest potential next steps or implications.
    
    Make the report informative, well-organized, and professional.
    """
    
    response = llm.invoke(prompt)
    
    return {
        **state,
        "final_report": response.content,
        "current_step": "report_generated"
    }


In [14]:
def quality_check_node(state: ResearchState) -> ResearchState:
    """Perform quality check on the generated report"""
    print("✅ Performing quality check...")
    
    prompt = f"""
    Review the following research report for quality and completeness:
    
    {state['final_report']}
    
    Provide a brief assessment (2-3 sentences) of the report quality and whether it 
    adequately addresses the original research question: "{state['question']}"
    
    Focus on:
    - Completeness of information
    - Logical structure
    - Relevance to research question
    """
    
    response = llm.invoke(prompt)
    
    print(f"\nQuality Assessment: {response.content}")
    
    return state  # No state modification, just quality check


In [15]:
def should_continue(state: ResearchState) -> str:
    """Router function to determine next step"""
    if state['current_step'] == 'start':
        return 'analyze_question'
    elif state['current_step'] == 'question_analyzed':
        return 'conduct_research'
    elif state['current_step'] == 'research_completed':
        return 'analyze_findings'
    elif state['current_step'] == 'analysis_completed':
        return 'generate_report'
    elif state['current_step'] == 'report_generated':
        return 'quality_check'
    else:
        return END


In [16]:
# Create the workflow graph
def create_research_assistant():
    """Create and return the research assistant graph"""
    
    workflow = StateGraph(ResearchState)
    
    # Add nodes
    workflow.add_node("analyze_question", question_analyzer_node)
    workflow.add_node("conduct_research", research_node)
    workflow.add_node("analyze_findings", analysis_node)
    workflow.add_node("generate_report", report_generation_node)
    workflow.add_node("quality_check", quality_check_node)
    
    # Set entry point
    workflow.set_entry_point("analyze_question")
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "analyze_question",
        should_continue
    )
    workflow.add_conditional_edges(
        "conduct_research", 
        should_continue
    )
    workflow.add_conditional_edges(
        "analyze_findings",
        should_continue
    )
    workflow.add_conditional_edges(
        "generate_report",
        should_continue
    )
    workflow.add_conditional_edges(
        "quality_check",
        lambda state: END  # Always end after quality check
    )
    
    return workflow.compile()

In [20]:
# 1. Compile the graph
app = create_research_assistant()

# 2. View the graph within a Jupyter Notebook or similar environment
print("Displaying graph visualization...")

# --- FIX IS HERE ---
# Use .draw_mermaid_png() instead of the deprecated .draw()
# This will return bytes of a PNG image
png_bytes = app.get_graph().draw_mermaid_png()

# Save the bytes to a file
file_path = "research_assistant_workflow.png"
with open(file_path, "wb") as f:
    f.write(png_bytes)

# Display the image inline in a Jupyter/IPython environment
if 'IPython' in globals():
    display(Image(png_bytes)) # Display directly from bytes
else:
    print(f"Graph saved to {file_path}. Open the file to view it.")


Displaying graph visualization...
Graph saved to research_assistant_workflow.png. Open the file to view it.


In [21]:
# Main execution function
def run_research_assistant(question: str) -> Dict[str, Any]:
    """
    Execute the research assistant workflow
    
    Args:
        question: The research question to investigate
        
    Returns:
        Dictionary containing the final state with research results
    """
    
    # Initialize the state
    initial_state = ResearchState(
        question=question,
        sub_questions=[],
        search_results={},
        analysis={},
        final_report="",
        current_step="start"
    )
    
    # Create and run the workflow
    research_agent = create_research_assistant()
    
    print(f"🚀 Starting research on: {question}")
    print("=" * 50)
    
    final_state = research_agent.invoke(initial_state)
    
    print("\n" + "=" * 50)
    print("🎯 Research completed!")
    
    return final_state


In [22]:
# Example usage and demonstration
if __name__ == "__main__":
    # Example research questions
    research_questions = [
        "The impact of artificial intelligence on healthcare",
        "Renewable energy trends in 2024",
        "The future of remote work technology"
    ]
    
    # Run research on the first question
    question = research_questions[0]
    results = run_research_assistant(question)
    
    # Display results
    print(f"\n📄 FINAL REPORT:")
    print("=" * 60)
    print(results['final_report'])
    
    # Print additional information
    print(f"\n📋 SUB-QUESTIONS ANALYZED:")
    for i, sq in enumerate(results['sub_questions'], 1):
        print(f"  {i}. {sq}")
    
    print(f"\n🔍 RESEARCH SOURCES: {len(results['search_results'])} sub-questions researched")
    print(f"📊 ANALYSIS COMPLETED: {len(results['analysis'])} sections analyzed")

🚀 Starting research on: The impact of artificial intelligence on healthcare
🔍 Analyzing research question...
📚 Conducting research...
  Researching: What are the key areas in healthcare where artificial intelligence is being utilized?
  Researching: What are the potential benefits of using artificial intelligence in healthcare?
  Researching: What are the potential risks or drawbacks of using artificial intelligence in healthcare?
  Researching: How is artificial intelligence changing the roles of healthcare professionals?
  Researching: What are the anticipated future developments in the use of artificial intelligence in healthcare?
📊 Analyzing research findings...
📝 Generating final report...
✅ Performing quality check...

Quality Assessment: The report is comprehensive and thorough, covering the major areas of AI application in healthcare, its benefits, risks, and the changing roles of healthcare professionals due to its adoption. The logical structure allows for easy understanding 

In [23]:
# Example usage and demonstration
if __name__ == "__main__":
    # Example research questions
    research_questions = [
        "The impact of artificial intelligence on healthcare",
        "Renewable energy trends in 2024",
        "The future of remote work technology"
    ]
    
    # Run research on the first question (Assuming 'run_research_assistant' is defined elsewhere)
    # The following code will likely raise a NameError if run without the function definition.
    # For demonstration purposes:
    # results = run_research_assistant(question) 

    # Since 'run_research_assistant' function is not defined here, 
    # I'll simulate some results for a complete example demonstration in markdown.
    
    # Simulated results structure:
    results = {
        'final_report': "This is a simulated final report discussing the impact of AI on healthcare, covering diagnostics, treatment personalization, and operational efficiency.",
        'sub_questions': [
            "How is AI used in medical diagnostics?",
            "What are the benefits of AI in personalized treatment plans?",
            "What challenges exist in AI implementation in hospitals?"
        ],
        'search_results': [
            {'query': 'query1', 'snippets': [...]},
            {'query': 'query2', 'snippets': [...]}
        ],
        'analysis': [
            {'section': 'Diagnostics'},
            {'section': 'Treatment Personalization'}
        ]
    }

    question = research_questions[0]
    
    # Display results
    print(f"\n📄 FINAL REPORT:")
    print("=" * 60)
    print(results['final_report'])
    
    # Print additional information
    print(f"\n📋 SUB-QUESTIONS ANALYZED:")
    for i, sq in enumerate(results['sub_questions'], 1):
        print(f"  {i}. {sq}")
    
    print(f"\n🔍 RESEARCH SOURCES: {len(results['search_results'])} sub-questions researched")
    print(f"📊 ANALYSIS COMPLETED: {len(results['analysis'])} sections analyzed")



📄 FINAL REPORT:
This is a simulated final report discussing the impact of AI on healthcare, covering diagnostics, treatment personalization, and operational efficiency.

📋 SUB-QUESTIONS ANALYZED:
  1. How is AI used in medical diagnostics?
  2. What are the benefits of AI in personalized treatment plans?
  3. What challenges exist in AI implementation in hospitals?

🔍 RESEARCH SOURCES: 2 sub-questions researched
📊 ANALYSIS COMPLETED: 2 sections analyzed
